# Supervised LoRA / QLoRA fine-tuning (Kaggle / Colab)

Fine-tune a Hugging Face causal language model with PEFT LoRA. Set `MODEL_ID` to a compatible Transformers causal LM. Data may be a Hugging Face dataset ID or a local CSV/JSON/JSONL file. This is not a universal trainer for all model architectures or Ollama models. Review model/dataset licenses, use a GPU runtime, and save checkpoints to persistent storage if the runtime is temporary. QLoRA requires a compatible CUDA runtime and `bitsandbytes`; disable 4-bit quantization to use regular LoRA.

In [ ]:
%pip -q install -U transformers datasets accelerate peft bitsandbytes

In [ ]:
import os, platform
from pathlib import Path
import torch

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"  # Hub repo ID or https://huggingface.co/owner/repo
DATASET_ID = ""
DATA_FILE = ""  # e.g. /kaggle/input/my-data/train.jsonl or /content/train.csv
TEXT_COLUMN = "text"  # use "instruction" and/or "response" for tabular SFT data
INSTRUCTION_COLUMN = "instruction"
RESPONSE_COLUMN = "response"
USE_4BIT = True
OUTPUT_DIR = "/kaggle/working/lora-adapter" if Path("/kaggle/working").exists() else "/content/lora-adapter"
from urllib.parse import urlparse
parsed_model = urlparse(MODEL_ID)
if parsed_model.scheme:
    if parsed_model.hostname not in {"huggingface.co", "www.huggingface.co"}:
        raise ValueError("Model URL must point to huggingface.co")
    model_parts = [part for part in parsed_model.path.split("/") if part]
    if len(model_parts) < 2: raise ValueError("Hub URL must identify an owner and repository")
    MODEL_ID = "/".join(model_parts[:2])
USE_HF_TOKEN = False  # set True for a gated/private model after adding HF_TOKEN to platform secrets
def platform_secret(name):
    if os.environ.get(name): return os.environ[name]
    try:
        from google.colab import userdata
    except ImportError: pass
    else: return userdata.get(name)
    try:
        from kaggle_secrets import UserSecretsClient
    except ImportError: pass
    else: return UserSecretsClient().get_secret(name)
    raise RuntimeError(f"Set {name} as an environment variable or notebook secret")
HF_TOKEN = platform_secret("HF_TOKEN") if USE_HF_TOKEN else os.environ.get("HF_TOKEN")
if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime in Kaggle/Colab before training")
print("GPU:", torch.cuda.get_device_name(0), "Python:", platform.python_version())
if bool(DATASET_ID) == bool(DATA_FILE):
    raise ValueError("Set exactly one of DATASET_ID or DATA_FILE")

## Load and format the training dataset

For instruction/response datasets, rows are converted to a simple prompt-completion text. For already-formatted datasets, set `TEXT_COLUMN` to the column containing each training example. Private or gated datasets may require authentication.

In [ ]:
from datasets import load_dataset

if DATASET_ID:
    dataset = load_dataset(DATASET_ID, split="train", token=HF_TOKEN)
else:
    extension = Path(DATA_FILE).suffix.lower()
    if extension == ".csv": loader = "csv"
    elif extension in {".json", ".jsonl"}: loader = "json"
    else: raise ValueError("DATA_FILE must be CSV, JSON, or JSONL")
    dataset = load_dataset(loader, data_files=DATA_FILE, split="train")
if len(dataset) == 0: raise ValueError("Training dataset is empty")
columns = set(dataset.column_names)
def format_example(row):
    if INSTRUCTION_COLUMN in columns and RESPONSE_COLUMN in columns:
        return {"text": f"### Instruction:\n{row[INSTRUCTION_COLUMN]}\n\n### Response:\n{row[RESPONSE_COLUMN]}"}
    if TEXT_COLUMN in columns:
        return {"text": str(row[TEXT_COLUMN])}
    raise ValueError(f"Dataset needs {TEXT_COLUMN!r} or both instruction/response columns")
dataset = dataset.map(format_example)
dataset = dataset.filter(lambda row: bool(row["text"].strip()))
if len(dataset) == 0: raise ValueError("No non-empty training examples")
print("Examples:", len(dataset), "Columns:", dataset.column_names)
print(dataset[0]["text"][:500])

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_TOKEN)
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
quantization_config = None
if USE_4BIT:
    if not torch.cuda.is_available(): raise RuntimeError("4-bit mode requires CUDA")
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
        bnb_4bit_use_double_quant=True,
    )
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, token=HF_TOKEN, quantization_config=quantization_config,
    torch_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    device_map="auto",
)
if USE_4BIT: model = prepare_model_for_kbit_training(model)
model.config.use_cache = False
lora_config = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05,
    target_modules="all-linear", bias="none", task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## Tokenize and train

Adjust sequence length, batch size, and training steps to fit the selected model and GPU. This basic causal-LM objective trains on all tokens in each formatted example; curate examples carefully and consider masking prompt tokens for instruction-only loss in production workflows.

In [ ]:
from transformers import DataCollatorForLanguageModeling, Trainer, TrainingArguments

MAX_LENGTH = 1024
def tokenize_batch(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH)
tokenized = dataset.map(tokenize_batch, batched=True, remove_columns=dataset.column_names)
tokenized = tokenized.filter(lambda row: len(row["input_ids"]) > 1)
if len(tokenized) == 0: raise ValueError("No examples remain after tokenization")
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    warmup_ratio=0.03,
    logging_steps=10,
    save_strategy="epoch",
    gradient_checkpointing=True,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    optim="paged_adamw_8bit" if USE_4BIT else "adamw_torch",
    report_to="none",
)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized,
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False),
)
trainer.train()
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("Adapter saved to", OUTPUT_DIR)

## Smoke test the adapter

The saved directory contains a PEFT adapter, not a standalone merged model. Load the same base model and adapter together for inference, or merge and export only if the base model license and available memory permit.

In [ ]:
model.eval()
inputs = tokenizer("### Instruction:\nExplain LoRA briefly.\n\n### Response:\n", return_tensors="pt").to(model.device)
with torch.inference_mode():
    generated = model.generate(**inputs, max_new_tokens=100, do_sample=True,
        temperature=0.7, top_p=0.9, pad_token_id=tokenizer.pad_token_id)
print(tokenizer.decode(generated[0], skip_special_tokens=True))